In [43]:
import os
import numpy as np


X_train = []
X_test = []
y_train = []
y_test = []

src_path = "/volatile/home/yb285618/SSF/declearn/examples/dreamt/data/dreamt_natural"

for client_folder in os.listdir(src_path):
    file = os.path.join(src_path, client_folder)
    X_train.append(np.load(os.path.join(file, "train_data.npy"), allow_pickle=False))
    X_test.append(np.load(os.path.join(file, "valid_data.npy"), allow_pickle=False))
    y_train.append(np.load(os.path.join(file, "train_target.npy"), allow_pickle=False))
    y_test.append(np.load(os.path.join(file, "valid_target.npy"), allow_pickle=False))


In [44]:
print(len(y_train))

21


In [45]:
X_train = np.concat(X_train, axis=0)
y_train = np.concat(y_train, axis=0)
print(X_train.shape)

(534715, 7, 64)


In [46]:
from sklearn.preprocessing import LabelEncoder

lb = LabelEncoder()
y_train_encoded = lb.fit_transform(y_train)

y_train_encoded

array([1, 4, 1, ..., 4, 4, 4], shape=(534715,))

In [47]:
import torch
import torch.nn as nn


class CNN(nn.Module):
    def __init__(self, channel_in=12, kernel_size=7):
        super().__init__()
        # 64 * 12
        self.conv1 = nn.Conv1d(
            in_channels=channel_in,
            out_channels=128,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )
        # => 64 * 64
        self.bn1 = nn.BatchNorm1d(128)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(32)
        self.bn4 = nn.BatchNorm1d(32)

        self.maxpool = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(
            in_channels=128,
            out_channels=64,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )

        self.conv3 = nn.Conv1d(
            in_channels=64,
            out_channels=32,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )

        self.conv4 = nn.Conv1d(
            in_channels=32,
            out_channels=32,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )
        # => 64 * 32

        self.fc1 = nn.Linear(32, 128)
        self.fc2 = nn.Linear(128, 5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu(x)
        x = x.mean(dim=2)
        x = self.relu(self.fc1(x))
        return self.fc2(x)


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import Lambda, Compose

torch.manual_seed(42)

transform = Compose(
    [
        torch.FloatTensor,
    ]
)


class DreamtDataset(Dataset):
    def __init__(self, X, y, transform=None, target_transform=None):
        super().__init__()
        self.X = X
        self.y = y
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, index):
        x = self.X[index]
        y = self.y[index]
        if self.transform:
            x = self.transform(self.X[index])

        if self.target_transform:
            y = self.target_transform(self.y[index])

        return x, y

    def __len__(self):
        return self.X.shape[0]


train_ds = DreamtDataset(X_train, y_train_encoded, transform=transform)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

In [49]:
sample, _ = next(iter(train_dl))

sample.dtype

torch.float32

In [50]:
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_model(model, train_dl, epochs, lr=0.001, device="cpu"):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss(reduction="sum")
    for epoch in tqdm(range(epochs)):
        model.train()
        empirical_risk = 0.0
        for X, y in train_dl:
            X = X.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            empirical_risk += loss.item()

        empirical_risk /= len(train_dl.dataset)
        print("Train loss: %.3f" % (empirical_risk))


model = CNN(channel_in=7)
model.to(DEVICE)

train_model(model, train_dl, epochs=5, device=DEVICE)

  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:19<01:19, 19.98s/it]

Train loss: 0.838


 40%|████      | 2/5 [00:40<01:00, 20.01s/it]

Train loss: 0.669


...

Train loss: 0.619


 80%|████████  | 4/5 [01:20<00:20, 20.13s/it]

Train loss: 0.590


100%|██████████| 5/5 [01:40<00:00, 20.12s/it]

Train loss: 0.570


In [51]:
from sklearn.metrics import f1_score


def test_model(model, test_dl, device="cpu"):
    criterion = nn.CrossEntropyLoss(reduction="sum")
    model.eval()
    generalization_error = 0.0
    correct = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X, y in test_dl:
            X = X.to(device)
            y = y.to(device)
            logits = model(X)
            loss = criterion(logits, y)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == y).sum().item()
            generalization_error += loss.item()
            all_preds.append(pred.cpu())
            all_targets.append(y.cpu())

        y_pred = torch.cat(all_preds).numpy()
        y_true = torch.cat(all_targets).numpy()
        generalization_error /= len(test_dl.dataset)
        accuracy = correct / len(test_dl.dataset)
        print(
            "Generalization Error: %.3f, Accuracy %.3f"
            % (generalization_error, accuracy)
        )

    return y_true, y_pred, generalization_error, accuracy


In [52]:
errors = []
accs = []
for client in range(len(X_test)):
    test_ds_client = DreamtDataset(
        X_test[client], lb.transform(y_test[client]), transform=transform
    )
    test_dl_client = DataLoader(test_ds_client, batch_size=128)
    y_true, y_pred, err, acc = test_model(model, test_dl_client, DEVICE)
    errors.append(err)
    accs.append(acc)

Generalization Error: 0.394, Accuracy 0.851
Generalization Error: 0.277, Accuracy 0.901
Generalization Error: 0.452, Accuracy 0.833
Generalization Error: 0.464, Accuracy 0.830
Generalization Error: 0.496, Accuracy 0.809
Generalization Error: 0.504, Accuracy 0.820
Generalization Error: 0.634, Accuracy 0.773
Generalization Error: 0.461, Accuracy 0.835
Generalization Error: 0.473, Accuracy 0.823
Generalization Error: 0.462, Accuracy 0.842
Generalization Error: 0.554, Accuracy 0.803
Generalization Error: 0.559, Accuracy 0.781
Generalization Error: 0.680, Accuracy 0.721
Generalization Error: 0.521, Accuracy 0.808
Generalization Error: 0.647, Accuracy 0.742
Generalization Error: 0.483, Accuracy 0.824
Generalization Error: 0.593, Accuracy 0.761
Generalization Error: 0.464, Accuracy 0.804
Generalization Error: 0.495, Accuracy 0.802
Generalization Error: 0.570, Accuracy 0.760
Generalization Error: 0.612, Accuracy 0.761


In [56]:
sum(errors) / len(accs)

0.5140776370308373

In [54]:
f1_score(y_true, y_pred, average="macro")

0.5980629255333769